<div style='background:linear-gradient(135deg,#1A2E4A 0%,#0D7377 100%);padding:50px 40px;border-radius:12px;color:white;text-align:center;font-family:Arial,sans-serif;'>
  <p style='font-size:13px;letter-spacing:3px;color:#14BDBD;margin:0 0 8px 0;'>AI / ML FOUNDATIONS COHORT</p>
  <h1 style='font-size:40px;margin:0 0 8px 0;font-weight:900;'>MEETING 6</h1>
  <h2 style='font-size:24px;font-weight:300;margin:0 0 30px 0;color:#D0D7E3;'>Classical ML II — Going Deeper</h2>
  <div style='width:60px;height:3px;background:#F0A500;margin:0 auto 30px auto;'></div>
  <p style='font-size:14px;color:#D0D7E3;margin:0 0 6px 0;'>KNN · Naive Bayes · SVM · Overfitting · Cross-Validation · Pipelines</p>
  <p style='font-size:13px;color:#6B8A9A;margin:0;'>Open in Jupyter or Google Colab</p>
</div>


---
## 🗺️ Session Roadmap

| Part | Topic | Time |
|------|-------|------|
| A | K-Nearest Neighbours (KNN) | 20 min |
| B | Naive Bayes | 20 min |
| C | Support Vector Machines (SVM) | 25 min |
| D | Overfitting vs Underfitting — Deep Treatment | 20 min |
| E | Cross-Validation & Hyperparameter Tuning | 20 min |
| F | Scikit-learn Pipelines | 15 min |
| G | Mini Tasks + Project | 20 min |

> 📖 Markdown = theory. ▶️ Code cells = live. ✏️ YOUR TURN = your challenge.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,
                      'axes.spines.right':False,'axes.grid':True,'grid.alpha':0.3})
sns.set_palette('muted')

# ── Shared dataset: Customer Churn ──────────────────────────────────────────
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_raw, y = make_classification(
    n_samples=600, n_features=8, n_informative=5,
    n_redundant=2, n_classes=2, random_state=42
)
feature_names = ['tenure','monthly_charge','num_products','support_calls',
                  'last_login_days','avg_session_min','contract_type','payment_method']
df = pd.DataFrame(X_raw, columns=feature_names)
df['churned'] = y

X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y)

scaler   = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_train)
X_te_sc  = scaler.transform(X_test)

print(f'Dataset: {df.shape[0]} customers, {df.shape[1]-1} features')
print(f'Churn rate: {df.churned.mean():.1%}')
print(f'Train: {len(X_train)}  |  Test: {len(X_test)}')
df.head()


---

# 🔵 Part A: K-Nearest Neighbours (KNN)
#### *Classify by similarity — no training, pure memory*

---

KNN is one of the most intuitive algorithms in ML. It makes **no assumptions** about  
the data distribution — it simply looks at the K closest training points to a new sample  
and takes a majority vote.

### How It Works
1. Store all training data (literally — no model is built)
2. For a new point: compute distance to every training point
3. Take the K nearest neighbours
4. Majority class wins (classification) or average value (regression)

$$d(\vec{a}, \vec{b}) = \sqrt{\sum_{i=1}^n (a_i - b_i)^2} \quad \text{(Euclidean distance)}$$

### Distance Metrics Compared
| Metric | Formula | Best For |
|--------|---------|----------|
| **Euclidean** | $\sqrt{\sum(a_i-b_i)^2}$ | Continuous, low-dimensional |
| **Manhattan** | $\sum|a_i-b_i|$ | High-dimensional, sparse |
| **Cosine** | $1 - \frac{a \cdot b}{\|a\|\|b\|}$ | Text, NLP, embeddings |

> ⚠️ **Critical:** KNN is extremely sensitive to feature scale.  
> A feature with range 0–100,000 will dominate a feature with range 0–1.  
> **Always standardise your features before using KNN.**


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# ── Distance computation — see what KNN actually does ────────────────────────
new_customer = X_te_sc[0]   # one test point
distances    = np.sqrt(np.sum((X_tr_sc - new_customer)**2, axis=1))
k            = 5
nearest_idx  = np.argsort(distances)[:k]

print('=== What KNN Does Internally ===')
print(f'Query point class: {y_test[0]}')
print(f'\n{k} nearest neighbours:')
print(f'  {"Idx":<6} {"Distance":<12} {"Class"}')
for idx in nearest_idx:
    print(f'  {idx:<6} {distances[idx]:<12.4f} {y_train[idx]}')
votes = y_train[nearest_idx]
pred  = int(votes.mean() >= 0.5)
print(f'\nVotes: {votes}  →  Majority: {pred}')
print(f'Correct: {pred == y_test[0]}')


In [ ]:
# ── KNN: Finding the right K ─────────────────────────────────────────────────
k_values = range(1, 31)
train_accs, test_accs = [], []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_tr_sc, y_train)
    train_accs.append(accuracy_score(y_train, knn.predict(X_tr_sc)))
    test_accs.append(accuracy_score(y_test,  knn.predict(X_te_sc)))

best_k   = k_values[np.argmax(test_accs)]
best_acc = max(test_accs)

plt.figure(figsize=(10, 4))
plt.plot(k_values, train_accs, 'o-', color='#0D7377', label='Train accuracy', linewidth=2)
plt.plot(k_values, test_accs,  's-', color='#F0A500', label='Test accuracy',  linewidth=2)
plt.axvline(best_k, color='red', linestyle='--', label=f'Best K={best_k}')
plt.xlabel('K (number of neighbours)')
plt.ylabel('Accuracy')
plt.title('KNN: Accuracy vs K — Finding the Sweet Spot', fontweight='bold')
plt.legend()
plt.tight_layout(); plt.show()

print(f'Best K: {best_k}  →  Test accuracy: {best_acc:.2%}')
print(f'K=1: train={train_accs[0]:.2%}, test={test_accs[0]:.2%}  ← classic overfitting')
print(f'K=30: train={train_accs[29]:.2%}, test={test_accs[29]:.2%}  ← possible underfitting')


In [ ]:
# ── Train final KNN and evaluate ─────────────────────────────────────────────
best_knn = KNeighborsClassifier(n_neighbors=best_k)
best_knn.fit(X_tr_sc, y_train)
knn_preds = best_knn.predict(X_te_sc)

print(f'=== KNN (K={best_k}) Classification Report ===')
print(classification_report(y_test, knn_preds, target_names=['Stayed','Churned']))


In [ ]:
# ✏️  YOUR TURN

# 1. Try Manhattan distance: KNeighborsClassifier(metric='manhattan')
#    Compare accuracy to Euclidean. Which is better on this dataset?
# 2. Try KNN WITHOUT scaling (use X_train, X_test directly)
#    Compare accuracy. Explain the difference in a comment.
# 3. Try KNeighborsClassifier(weights='distance') — what does this do?
# 4. Plot a confusion matrix for best_knn using ConfusionMatrixDisplay

# ── Write below ──────────────────────────────────────────────


---

# 🎲 Part B: Naive Bayes
#### *Probabilistic classification using Bayes theorem*

---

Naive Bayes applies Bayes theorem to classify data. It is called **naive** because  
it assumes all features are **conditionally independent** given the class label —  
a simplification that is almost never true but works remarkably well in practice.

$$P(\text{class} \mid \vec{x}) \propto P(\text{class}) \prod_{i=1}^n P(x_i \mid \text{class})$$

For each class, it multiplies together the probability of each feature given that class,  
then picks the class with the highest result.

### Three Variants
| Variant | Assumes Feature Distribution | Best For |
|---------|------------------------------|----------|
| **GaussianNB** | Normal (Gaussian) | Continuous features |
| **MultinomialNB** | Counts / frequencies | Text (word counts) |
| **BernoulliNB** | Binary (0/1) | Binary features, text presence |

### Why It Works Despite the Naive Assumption
Even though features are rarely independent, the **ranking** of class probabilities  
is often correct. The model does not need exact probabilities — just the right winner.


In [ ]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.preprocessing import MinMaxScaler

# Gaussian NB on our customer data
gnb   = GaussianNB()
gnb.fit(X_train, y_train)
gnb_preds = gnb.predict(X_test)
gnb_probs = gnb.predict_proba(X_test)

print('=== Gaussian Naive Bayes ===')
print(f'Accuracy: {accuracy_score(y_test, gnb_preds):.2%}')
print(f'\nLearned class priors:')
for cls, prior in enumerate(gnb.class_prior_):
    print(f'  P(class={cls}) = {prior:.3f}')

print(f'\nLearned feature means per class (first 3 features):')
print(f'  {"Feature":<22} {"Class 0 mean":>15} {"Class 1 mean":>15}')
for i, name in enumerate(feature_names[:3]):
    print(f'  {name:<22} {gnb.theta_[0][i]:>15.3f} {gnb.theta_[1][i]:>15.3f}')

print(f'\nSample prediction probabilities (first 5 test points):')
for i in range(5):
    print(f'  True: {y_test[i]}  | P(0)={gnb_probs[i][0]:.3f} P(1)={gnb_probs[i][1]:.3f}'
          f' → Pred: {gnb_preds[i]}')


In [ ]:
# ── Naive Bayes on Text — Spam Detection ────────────────────────────────────
from sklearn.feature_extraction.text import CountVectorizer

# Small synthetic SMS dataset
messages = [
    ('You have WON a FREE iPhone! Click NOW to claim your prize!!!', 1),
    ('Congratulations! You are selected for a cash reward of $5000', 1),
    ('URGENT: Your account will be suspended. Verify your details now', 1),
    ('FREE entry to win! Text WIN to 87121 NOW', 1),
    ('Hey, are we still on for lunch tomorrow?', 0),
    ('Can you send me the meeting notes from today please', 0),
    ('I will be late to the office, stuck in traffic', 0),
    ('The project report is ready, let me know when to send it', 0),
    ('Buy now and save 80%! Limited time offer!', 1),
    ('Did you see the game last night? What a match!', 0),
    ('Your loan has been approved! Claim $10,000 instantly', 1),
    ('Reminder: team standup at 9am tomorrow', 0),
]
texts, labels = zip(*messages)

vectorizer = CountVectorizer(stop_words='english')
X_text     = vectorizer.fit_transform(texts)

# MinMax scaling for MultinomialNB (needs non-negative)
mnb = MultinomialNB()
mnb.fit(X_text, labels)

print('=== Naive Bayes Spam Classifier ===')
test_msgs = [
    'You have won a free prize click here',
    'Can we reschedule the meeting to Friday',
    'Claim your FREE cash reward immediately',
    'Are you coming to the workshop next week',
]
X_test_text = vectorizer.transform(test_msgs)
preds = mnb.predict(X_test_text)
probs = mnb.predict_proba(X_test_text)

for msg, pred, prob in zip(test_msgs, preds, probs):
    label = '🚨 SPAM' if pred==1 else '✅ HAM '
    print(f'{label} ({prob[1]:.0%} spam prob): "{msg}"')


---

# ⚔️ Part C: Support Vector Machines (SVM)
#### *Finding the widest possible decision boundary*

---

SVM finds the **hyperplane** (decision boundary) that maximises the **margin** —  
the distance between the boundary and the nearest data points of each class.  
Those nearest points are called **support vectors** and they alone define the boundary.

$$\text{Maximise: } \frac{2}{\|\vec{w}\|} \quad \text{Subject to: } y_i(\vec{w} \cdot \vec{x}_i + b) \geq 1$$

### The Kernel Trick — Handling Non-Linear Data
Real data is rarely linearly separable. SVM solves this by **mapping data into a higher-dimensional  
space** where it becomes separable — without explicitly computing the transformation.

$$K(\vec{x}_i, \vec{x}_j) = \phi(\vec{x}_i) \cdot \phi(\vec{x}_j)$$

| Kernel | Formula | Use Case |
|--------|---------|----------|
| **Linear** | $\vec{x}_i \cdot \vec{x}_j$ | Linearly separable, high-dimensional text |
| **RBF (Gaussian)** | $\exp(-\gamma\|\vec{x}_i-\vec{x}_j\|^2)$ | Most general purpose |
| **Polynomial** | $(\vec{x}_i \cdot \vec{x}_j + r)^d$ | Image recognition, NLP |

### Key Hyperparameters
| Parameter | Controls | Small value | Large value |
|-----------|---------|-------------|-------------|
| **C** | Margin vs misclassification tradeoff | Wider margin, more errors | Narrower margin, fewer errors |
| **gamma** (RBF) | How far influence of each point reaches | Smooth boundary | Complex, tight boundary |


In [ ]:
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

# ── SVM decision boundaries: Linear vs RBF ───────────────────────────────────
from sklearn.datasets import make_circles, make_moons

datasets = [
    ('Linearly Separable', *make_classification(n_samples=200, n_features=2,
      n_informative=2, n_redundant=0, random_state=1)),
    ('Moons (Non-linear)',  *make_moons(n_samples=200, noise=0.15, random_state=42)),
    ('Circles (Non-linear)',*make_circles(n_samples=200, noise=0.1,
      factor=0.5, random_state=42)),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for col, (name, X_d, y_d) in enumerate(datasets):
    for row, (kernel, title) in enumerate([('linear','Linear kernel'),('rbf','RBF kernel')]):
        ax = axes[row][col]
        svm = SVC(kernel=kernel, C=1.0, gamma='scale')
        svm.fit(X_d, y_d)

        # Decision boundary
        h  = 0.02
        x_min,x_max = X_d[:,0].min()-0.5, X_d[:,0].max()+0.5
        y_min,y_max = X_d[:,1].min()-0.5, X_d[:,1].max()+0.5
        xx,yy = np.meshgrid(np.arange(x_min,x_max,h), np.arange(y_min,y_max,h))
        Z = svm.predict(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape)
        ax.contourf(xx,yy,Z,alpha=0.25,cmap='RdYlGn')
        ax.scatter(X_d[:,0],X_d[:,1],c=y_d,cmap='RdYlGn',edgecolors='k',s=30)

        # Support vectors
        sv = svm.support_vectors_
        ax.scatter(sv[:,0],sv[:,1],s=120,facecolors='none',
                   edgecolors='navy',linewidths=2,label='Support vectors')
        acc = accuracy_score(y_d, svm.predict(X_d))
        ax.set_title(f'{name}\n{title} — Acc={acc:.0%}', fontweight='bold', fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('SVM Decision Boundaries: Linear vs RBF Kernel', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ── SVM on customer churn — effect of C and gamma ────────────────────────────
results = []
for C in [0.01, 0.1, 1, 10, 100]:
    for gamma in ['scale', 0.001, 0.01, 0.1]:
        svm = SVC(kernel='rbf', C=C, gamma=gamma, probability=True)
        svm.fit(X_tr_sc, y_train)
        tr_acc = accuracy_score(y_train, svm.predict(X_tr_sc))
        te_acc = accuracy_score(y_test,  svm.predict(X_te_sc))
        results.append({'C':C,'gamma':str(gamma),'train_acc':tr_acc,'test_acc':te_acc})

res_df = pd.DataFrame(results)
best   = res_df.loc[res_df.test_acc.idxmax()]
print('=== SVM Hyperparameter Effect ===')
print(res_df.sort_values('test_acc', ascending=False).head(8).to_string(index=False))
print(f'\nBest: C={best.C}, gamma={best.gamma}  →  Test: {best.test_acc:.2%}  Train: {best.train_acc:.2%}')

# Final SVM
final_svm = SVC(kernel='rbf', C=float(best.C),
                 gamma=best.gamma if best.gamma!='scale' else 'scale',
                 probability=True)
final_svm.fit(X_tr_sc, y_train)
svm_preds = final_svm.predict(X_te_sc)
print(f'\n=== Final SVM Classification Report ===')
print(classification_report(y_test, svm_preds, target_names=['Stayed','Churned']))


---

## ⏱️ Mini Task 1 — *Algorithm Showdown*

> **10 minutes** — solo or pairs. Run, interpret, share one finding.

---


Using the **churn dataset** already loaded, train all three algorithms and compare:

| Model | Train Acc | Test Acc | Precision | Recall | F1 |
|-------|-----------|----------|-----------|--------|----|
| KNN (best K) | ? | ? | ? | ? | ? |
| Naive Bayes | ? | ? | ? | ? | ? |
| SVM (RBF) | ? | ? | ? | ? | ? |

Fill the table. Then answer: which would you recommend for a real churn prediction product, and why?


In [ ]:
# ✏️  YOUR TURN

# from sklearn.metrics import f1_score, precision_score, recall_score
# models = {
#     'KNN':       KNeighborsClassifier(n_neighbors=best_k),
#     'NaiveBayes':GaussianNB(),
#     'SVM':       SVC(kernel='rbf', C=1.0, gamma='scale'),
# }
# results = []
# for name, model in models.items():
#     model.fit(X_tr_sc, y_train)
#     preds = model.predict(X_te_sc)
#     # Append dict with all metrics
#     pass
# print(pd.DataFrame(results))
# # Your recommendation (as a comment):

# ── Write below ──────────────────────────────────────────────


---

# ⚖️ Part D: Overfitting vs Underfitting
#### *The central tension in all of Machine Learning*

---

Understanding overfitting and underfitting at a deep level is what makes you  
a practitioner rather than someone who just runs notebooks.

### The Bias-Variance Tradeoff

Every model error can be decomposed into three components:

$$\text{Total Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Noise}$$

| Component | Meaning | High = |
|-----------|---------|--------|
| **Bias** | How wrong the model is on average | Underfitting — too simple |
| **Variance** | How much the model changes with different data | Overfitting — too complex |
| **Irreducible noise** | Inherent randomness in data | Cannot be eliminated |

You cannot reduce both simultaneously — reducing one increases the other.  
Your job is to find the **optimal complexity** that minimises total error.

### How to Detect Each
| Symptom | Likely Cause |
|---------|-------------|
| Train acc = 99%, Test acc = 62% | Overfitting |
| Train acc = 63%, Test acc = 61% | Underfitting |
| Both around 90% | Well fitted |
| Test acc higher than train acc | Unusual — check data leakage |


In [ ]:
# ── Visualising bias-variance tradeoff across model complexities ─────────────
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score

np.random.seed(42)
X_1d = np.sort(np.random.uniform(-3, 3, 80))
y_1d = np.sin(X_1d) + 0.3*np.random.randn(80)
X_1d_r = X_1d.reshape(-1,1)

degrees   = [1, 3, 8, 15]
X_plot    = np.linspace(-3, 3, 200).reshape(-1,1)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, deg in zip(axes, degrees):
    model = Pipeline([('poly', PolynomialFeatures(deg)), ('reg', LinearRegression())])
    model.fit(X_1d_r, y_1d)
    train_mse = mean_squared_error(y_1d, model.predict(X_1d_r))
    cv_score  = -cross_val_score(model, X_1d_r, y_1d, cv=5,
                                  scoring='neg_mean_squared_error').mean()
    ax.scatter(X_1d, y_1d, alpha=0.4, color='#0D7377', s=25)
    ax.plot(X_plot, model.predict(X_plot), color='#F0A500', linewidth=2.5)
    ax.set_title(f'Degree {deg}\nTrain MSE={train_mse:.3f}\nCV MSE={cv_score:.3f}',
                 fontweight='bold', fontsize=9)
    ax.set_ylim(-2.5, 2.5)

plt.suptitle('Polynomial Degree vs Bias-Variance Tradeoff', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print('Degree 1: underfits (high bias)  |  Degree 15: overfits (high variance)')
print('Degree 3: best — low CV error, captures the true sine pattern')


In [ ]:
# ── Regularisation: L1 (Lasso) vs L2 (Ridge) ─────────────────────────────────
# Regularisation adds a penalty term to the loss to prevent large weights
# L2 (Ridge): loss + lambda * sum(w^2)   ← shrinks weights, keeps all
# L1 (Lasso): loss + lambda * sum(|w|)  ← drives some weights to exactly 0

from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.preprocessing import PolynomialFeatures

# High-degree polynomial that will overfit without regularisation
deg     = 12
poly    = PolynomialFeatures(deg)
X_poly  = poly.fit_transform(X_1d_r)

models  = [
    ('No Reg (overfit)', LinearRegression(), '#E74C3C'),
    ('L2 Ridge (alpha=1)', Ridge(alpha=1.0), '#F0A500'),
    ('L2 Ridge (alpha=10)', Ridge(alpha=10.0), '#0D7377'),
    ('L1 Lasso (alpha=0.01)', Lasso(alpha=0.01, max_iter=5000), '#8E44AD'),
]

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(X_1d, y_1d, alpha=0.4, color='gray', s=25, label='Data')
X_plot_poly = poly.transform(X_plot)

print(f'{'Model':<28} {'Train MSE':>12} {'# non-zero weights':>20}')
print('-'*62)
for name, m, color in models:
    m.fit(X_poly, y_1d)
    train_mse  = mean_squared_error(y_1d, m.predict(X_poly))
    coef       = m.coef_ if hasattr(m,'coef_') else []
    nonzero    = np.sum(np.abs(coef) > 1e-4)
    print(f'{name:<28} {train_mse:>12.4f} {nonzero:>20}')
    y_plot = m.predict(X_plot_poly)
    ax.plot(X_plot, np.clip(y_plot,-3,3), color=color, linewidth=2, label=name)

ax.set_ylim(-2.5, 2.5)
ax.legend(fontsize=8)
ax.set_title('Effect of L1/L2 Regularisation on Degree-12 Polynomial', fontweight='bold')
plt.tight_layout(); plt.show()
print('\nL1 drives weights to zero (sparse). L2 shrinks all weights. Both reduce overfitting.')


---

# 🔬 Part E: Cross-Validation & Hyperparameter Tuning
#### *Evaluating and improving models honestly*

---

### Why a Single Train/Test Split Is Not Enough
A single split is **lucky or unlucky** depending on which samples ended up where.  
If test set happens to contain easy samples, your accuracy is inflated.  
Cross-validation averages over many splits to get a **stable, honest estimate**.

### K-Fold Cross-Validation
Split data into K equal folds. Train on K-1, test on the remaining 1. Rotate K times.

$$\text{CV Score} = \frac{1}{K}\sum_{k=1}^K \text{score}(\text{fold}_k)$$

### Stratified K-Fold
Ensures each fold has the **same class distribution** as the original dataset.  
Essential for imbalanced datasets — always use this for classification.

### GridSearchCV vs RandomizedSearchCV
| Method | How | When |
|--------|-----|------|
| **GridSearchCV** | Try every combination | Small, well-defined grids |
| **RandomizedSearchCV** | Sample random combinations | Large grids, faster |
| **BayesianOptimization** | Smart sequential search | Production models |


In [ ]:
from sklearn.model_selection import (cross_val_score, StratifiedKFold,
                                        GridSearchCV, RandomizedSearchCV,
                                        learning_curve)
from sklearn.ensemble import RandomForestClassifier

# ── Cross-validation deep comparison ─────────────────────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('=== 5-Fold Stratified Cross-Validation ===')
print(f'  {"Model":<20} {"Mean Acc":>10} {"Std":>8} {"Min":>8} {"Max":>8}')
print('  '+'-'*56)

for name, model in [
    ('KNN k=5',    KNeighborsClassifier(n_neighbors=5)),
    ('Naive Bayes', GaussianNB()),
    ('SVM RBF',    SVC(kernel='rbf', C=1.0, gamma='scale')),
    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42)),
]:
    scores = cross_val_score(model, X_tr_sc, y_train,
                              cv=skf, scoring='accuracy')
    print(f'  {name:<20} {scores.mean():>10.3f} {scores.std():>8.3f}'
          f' {scores.min():>8.3f} {scores.max():>8.3f}')
print('\nHigh std = model is sensitive to which data it trains on (unstable)')


In [ ]:
# ── Learning Curves — diagnosing overfitting and data hunger ────────────────
rf = RandomForestClassifier(n_estimators=100, random_state=42)
train_sizes, train_scores, val_scores = learning_curve(
    rf, X_tr_sc, y_train,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5, scoring='accuracy', n_jobs=-1
)

tr_mean, tr_std = train_scores.mean(1), train_scores.std(1)
va_mean, va_std = val_scores.mean(1),   val_scores.std(1)

plt.figure(figsize=(9, 5))
plt.plot(train_sizes, tr_mean, 'o-', color='#0D7377', label='Training score', lw=2)
plt.fill_between(train_sizes, tr_mean-tr_std, tr_mean+tr_std, alpha=0.15, color='#0D7377')
plt.plot(train_sizes, va_mean, 's-', color='#F0A500', label='CV score', lw=2)
plt.fill_between(train_sizes, va_mean-va_std, va_mean+va_std, alpha=0.15, color='#F0A500')
plt.xlabel('Training set size'); plt.ylabel('Accuracy')
plt.title('Learning Curve — Random Forest', fontweight='bold')
plt.legend(); plt.tight_layout(); plt.show()

gap = tr_mean[-1] - va_mean[-1]
print(f'Final train-val gap: {gap:.3f}')
if gap > 0.1:
    print('Large gap → overfitting. Try: more data, regularisation, simpler model.')
elif va_mean[-1] < 0.75:
    print('Both scores low → underfitting. Try: more features, more complex model.')
else:
    print('Small gap, decent accuracy → well fitted!')


In [ ]:
# ── GridSearchCV: Professional hyperparameter tuning ─────────────────────────
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [None, 5, 10],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2'],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, cv=5, scoring='f1',
    n_jobs=-1, verbose=0, return_train_score=True
)
grid_search.fit(X_tr_sc, y_train)

print('=== GridSearchCV Results ===')
print(f'Best parameters: {grid_search.best_params_}')
print(f'Best CV F1:      {grid_search.best_score_:.4f}')

best_rf = grid_search.best_estimator_
rf_preds = best_rf.predict(X_te_sc)
print(f'Test accuracy:   {accuracy_score(y_test, rf_preds):.2%}')
print(f'\n=== Final Classification Report ===')
print(classification_report(y_test, rf_preds, target_names=['Stayed','Churned']))

# Feature importance from best model
importances = pd.Series(best_rf.feature_importances_, index=feature_names).sort_values()
plt.figure(figsize=(8, 4))
importances.plot(kind='barh', color='#0D7377', edgecolor='white')
plt.title('Feature Importances — Best Random Forest', fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout(); plt.show()


---

# 🔗 Part F: Scikit-learn Pipelines
#### *Professional, leakage-free, reproducible ML workflows*

---

A **Pipeline** chains preprocessing steps and a model into a single object.  
This is not just clean code — it **prevents data leakage**, makes deployment trivial,  
and enables end-to-end cross-validation and hyperparameter tuning.

### The Data Leakage Problem Without Pipelines
```python
# WRONG — leaks test data into the scaler
scaler = StandardScaler()
X_all_scaled = scaler.fit_transform(X)   # fitted on ALL data including test!
X_train, X_test = train_test_split(X_all_scaled, ...)
```
```python
# CORRECT — scaler only sees training data
pipeline = Pipeline([('scaler', StandardScaler()), ('model', SVC())])
pipeline.fit(X_train, y_train)   # scaler fitted ONLY on train
pipeline.predict(X_test)         # scaler applied (not refitted) to test
```
Even a small amount of test data leaking into preprocessing can inflate performance metrics  
significantly — making you think your model is better than it is.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
import joblib

# ── A complete production-grade pipeline ─────────────────────────────────────
# Simulate mixed data: some numeric, some categorical, some missing
np.random.seed(42)
n = 400
data = pd.DataFrame({
    'age':        np.random.randint(20, 65, n).astype(float),
    'income':     np.random.normal(55000, 20000, n),
    'score':      np.random.uniform(300, 850, n),
    'region':     np.random.choice(['North','South','East','West'], n),
    'plan':       np.random.choice(['Basic','Standard','Premium'], n),
    'churned':    np.random.binomial(1, 0.3, n)
})
# Inject missings
data.loc[np.random.choice(n,30,replace=False),'age']    = np.nan
data.loc[np.random.choice(n,20,replace=False),'income'] = np.nan

X_p = data.drop('churned', axis=1)
y_p = data['churned']
X_tr_p, X_te_p, y_tr_p, y_te_p = train_test_split(X_p, y_p, test_size=0.2, random_state=42)

# Define separate preprocessors for numeric and categorical
numeric_features     = ['age','income','score']
categorical_features = ['region','plan']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer,     numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Full pipeline
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

full_pipeline.fit(X_tr_p, y_tr_p)
preds_p = full_pipeline.predict(X_te_p)

print('=== Full Pipeline with Mixed Data ===')
print(f'Accuracy: {accuracy_score(y_te_p, preds_p):.2%}')
print(classification_report(y_te_p, preds_p, target_names=['Stayed','Churned']))

# Save and reload the entire pipeline
joblib.dump(full_pipeline, '/tmp/churn_pipeline.pkl')
loaded = joblib.load('/tmp/churn_pipeline.pkl')
reloaded_preds = loaded.predict(X_te_p)
print(f'Reloaded pipeline accuracy: {accuracy_score(y_te_p, reloaded_preds):.2%} (matches: {(preds_p==reloaded_preds).all()})')


In [ ]:
# ── GridSearchCV on full pipeline — tune both preprocessing and model ────────
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# With pipelines, param names use __ notation: step__param
param_dist = {
    'preprocessor__num__imputer__strategy': ['median','mean'],
    'model__n_estimators':   randint(50, 200),
    'model__max_depth':      [None, 5, 10, 15],
    'model__min_samples_split': randint(2, 15),
}

rand_search = RandomizedSearchCV(
    full_pipeline, param_dist,
    n_iter=20, cv=5, scoring='f1',
    random_state=42, n_jobs=-1
)
rand_search.fit(X_tr_p, y_tr_p)

print('=== RandomizedSearchCV on Full Pipeline ===')
print(f'Best params: {rand_search.best_params_}')
print(f'Best CV F1:  {rand_search.best_score_:.4f}')
print(f'Test acc:    {accuracy_score(y_te_p, rand_search.best_estimator_.predict(X_te_p)):.2%}')


---

## ⏱️ Mini Task 2 — *Build & Tune a Full Pipeline*

> **15 minutes** — solo or pairs. Run, interpret, share one finding.

---


Build a complete pipeline for the customer churn dataset:

1. Impute any missing values, scale numeric features
2. Choose any model: KNN, Naive Bayes, SVM, or Random Forest
3. Tune at least 2 hyperparameters using `GridSearchCV` or `RandomizedSearchCV`
4. Evaluate on test set — report accuracy, precision, recall, F1
5. Save the pipeline with `joblib.dump()`
6. Reload and verify predictions match


In [ ]:
# ✏️  YOUR TURN

# # Your full pipeline here
# from sklearn.pipeline import Pipeline
# from sklearn.impute import SimpleImputer
# # ...
# 
# my_pipeline = Pipeline([
#     ('scaler', ...),
#     ('model',  ...)
# ])
# # Tune with GridSearchCV or RandomizedSearchCV
# # Evaluate, save, reload

# ── Write below ──────────────────────────────────────────────


---
## ✅ What You Covered in Meeting 6
| Concept | Depth |
|---------|-------|
| KNN — distance computation, K selection, distance metrics | Deep |
| Naive Bayes — Gaussian and Multinomial, spam demo | Deep |
| SVM — margin, kernel trick, C and gamma tuning | Deep |
| Overfitting vs underfitting — bias-variance, learning curves | Deep |
| Regularisation — L1 and L2, visualised | Deep |
| Cross-validation — stratified K-fold, learning curves | Deep |
| GridSearchCV + RandomizedSearchCV | Deep |
| Scikit-learn Pipelines — leakage, mixed data, save/load | Deep |

> 🔭 **Next — Meeting 7:** Unsupervised Learning — K-Means, DBSCAN, PCA, t-SNE, anomaly detection.

<div style='background:linear-gradient(135deg,#1A2E4A 0%,#0D7377 100%);padding:28px;border-radius:10px;color:white;text-align:center;'>
  <h3 style='margin:0 0 8px 0;color:#14BDBD;'>You now know why models fail — and how to fix them.</h3>
  <p style='color:#F0A500;font-weight:bold;margin:0;'>See you in Meeting 7. 🚀</p>
</div>
